# T-08 Validation — lxyuan sentiment model spread check

**Purpose:** Confirm that the lxyuan/distilbert-base-multilingual-cased-sentiments-student
replacement for VADER produces meaningfully wider `odi_score` spread across clusters on
real synthetic datasets.

**Benchmark reference (from decisions.md):**
- VADER spread (avg_sentiment): 0.4398 — compressed on calm B2B text
- lxyuan spread (avg_sentiment): 0.6446 — best model in benchmark

**Pass criteria:** `odi_score` std > 0.10 on at least two of three datasets.
If any dataset fails, the lxyuan integration mapping (`_score_to_compound`) must be
inspected before T-16 proceeds.

**Run order:** Run all cells top to bottom. Results summary at the bottom.

## 0. Setup — imports and pipeline path

In [1]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Add the repo root to the path so pipeline/* is importable
REPO_ROOT = os.path.abspath('..')   # adjust if you run from a different location
sys.path.insert(0, REPO_ROOT)

import pandas as pd
import numpy as np
from pathlib import Path

from pipeline.chunker import chunk_text
from pipeline.embedder import embed_chunks
from pipeline.clusterer import cluster
from pipeline.odi_scorer import score_clusters

print('All imports OK')
print(f'Repo root: {REPO_ROOT}')

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 6827.37it/s]


All imports OK
Repo root: /Users/lucashome/capstone/discovery-lens


## 1. Loaders — one function per file type

These loaders replicate what `extractor.py` + `chunker.py` do in the app,
but work directly on disk paths so the notebook is self-contained.
Each file is mapped to its correct `source_type`.

In [2]:
def load_txt(path: str, source_type: str) -> list[dict]:
    """Load a .txt interview/usability file and chunk it."""
    raw = Path(path).read_text(encoding='utf-8', errors='replace')
    filename = Path(path).name
    return chunk_text(raw, filename, source_type)


def load_csv_text_col(path: str, text_col: str, source_type: str,
                      max_rows: int = 80) -> list[dict]:
    """
    Load a CSV, concatenate all values in `text_col` into one string,
    and chunk it. max_rows caps the input so we don't exceed the
    500-chunk session ceiling during validation.
    """
    df = pd.read_csv(path, nrows=max_rows)
    if text_col not in df.columns:
        raise ValueError(f'{Path(path).name}: column "{text_col}" not found. '
                         f'Available: {list(df.columns)}')
    # Drop nulls and join into one blob — chunker splits on sentences
    joined = ' '.join(df[text_col].dropna().astype(str).tolist())
    filename = Path(path).name
    return chunk_text(joined, filename, source_type)


def load_csv_ticket(path: str, text_col: str, source_type: str = 'ticket',
                    max_rows: int = 40) -> list[dict]:
    """Convenience wrapper for ticket CSVs (customer_message / text column)."""
    return load_csv_text_col(path, text_col, source_type, max_rows)


print('Loaders defined')

Loaders defined


## 2. Run pipeline for each dataset

### 2a. Lidl Plus

Files available:
- `interview_lidl_01–05.txt` → `interview`
- `usability_lidl_01–04.txt` → `usability`
- `lidl_plus_reviews.csv` (column: `content`) → `review`
- `tickets_lidl.csv` (column: `text`) → `ticket`

In [3]:
DATA = Path(REPO_ROOT) / 'data' / 'synthetic'

lidl_dir = DATA / 'lidl_plus_app'
lidl_chunks = []

# Interviews
for i in range(1, 6):
    p = lidl_dir / f'interview_lidl_0{i}.txt'
    if p.exists():
        lidl_chunks += load_txt(str(p), 'interview')

# Usability notes
for i in range(1, 5):
    p = lidl_dir / f'usability_lidl_0{i}.txt'
    if p.exists():
        lidl_chunks += load_txt(str(p), 'usability')

# Reviews CSV
lidl_chunks += load_csv_text_col(
    str(lidl_dir / 'lidl_plus_reviews.csv'), 'content', 'review', max_rows=60
)

# Tickets CSV
lidl_chunks += load_csv_ticket(
    str(lidl_dir / 'tickets_lidl.csv'), 'text', max_rows=40
)

print(f'Lidl chunks: {len(lidl_chunks)}')
print(f'Source types present: {set(c["source_type"] for c in lidl_chunks)}')

Lidl chunks: 431
Source types present: {'usability', 'interview', 'review', 'ticket'}


In [4]:
# Embed + cluster + score
print('Embedding Lidl chunks (this takes ~30s on first run)...')
lidl_emb = embed_chunks(lidl_chunks)
lidl_clusters = cluster(lidl_chunks, lidl_emb)
lidl_scored = score_clusters(lidl_clusters, lidl_chunks)

print(f'Clusters: {len(lidl_clusters)}')
pd.DataFrame(lidl_scored)[[
    'cluster_id', 'cluster_size', 'avg_sentiment',
    'satisfaction', 'source_type_diversity', 'odi_score',
    'evidence_robustness', 'priority_score'
]]

Embedding Lidl chunks (this takes ~30s on first run)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8046.89it/s]


Clusters: 7


,cluster_id,cluster_size,avg_sentiment,satisfaction,source_type_diversity,odi_score,evidence_robustness,priority_score
0,1,76,-0.0998,0.4501,0.6667,0.0970,0.4951,0.2562
1,2,86,0.1080,0.5540,0.6667,0.0890,0.5032,0.2547
2,5,65,-0.1479,0.4260,0.6667,0.0866,0.4861,0.2464
3,0,80,0.0874,0.5437,0.5000,0.0847,0.3900,0.2068
4,6,59,0.2458,0.6229,0.5000,0.0516,0.3729,0.1801
5,3,40,0.1832,0.5916,0.5000,0.0379,0.3575,0.1657
6,4,25,0.0698,0.5349,0.3333,0.0270,0.2370,0.1110


### 2b. Asana

Files available:
- `interview_asana_01–04.txt` → `interview`
- `usability_asana_01–03.txt` → `usability`
- `reviews_asana.csv` (column: `text`) → `review`
- `tickets_asana.csv` (column: `text`) → `ticket`
- `reddit_asana.csv` (column: `post_text`) → `social`
- `sales_notes_asana.txt` → `internal`

In [6]:
asana_dir = DATA / 'asana'
asana_chunks = []

# Interviews
for i in range(1, 5):
    p = asana_dir / f'interview_asana_0{i}.txt'
    if p.exists():
        asana_chunks += load_txt(str(p), 'interview')

# Usability notes
for i in range(1, 4):
    p = asana_dir / f'usability_asana_0{i}.txt'
    if p.exists():
        asana_chunks += load_txt(str(p), 'usability')

# Reviews CSV
asana_chunks += load_csv_text_col(
    str(asana_dir / 'reviews_asana.csv'), 'text', 'review', max_rows=60
)

# Tickets CSV
asana_chunks += load_csv_ticket(
    str(asana_dir / 'tickets_asana.csv'), 'text', max_rows=40
)

# Reddit threads — social source type
asana_chunks += load_csv_text_col(
    str(asana_dir / 'reddit_asana.csv'), 'post_text', 'social', max_rows=30
)

# Sales notes — internal
p = asana_dir / 'sales_notes_asana.txt'
if p.exists():
    asana_chunks += load_txt(str(p), 'internal')

print(f'Asana chunks: {len(asana_chunks)}')
print(f'Source types present: {set(c["source_type"] for c in asana_chunks)}')

Asana chunks: 419
Source types present: {'interview', 'review', 'social', 'internal', 'usability', 'ticket'}


In [7]:
print('Embedding Asana chunks...')
asana_emb = embed_chunks(asana_chunks)
asana_clusters = cluster(asana_chunks, asana_emb)
asana_scored = score_clusters(asana_clusters, asana_chunks)

print(f'Clusters: {len(asana_clusters)}')
pd.DataFrame(asana_scored)[[
    'cluster_id', 'cluster_size', 'avg_sentiment',
    'satisfaction', 'source_type_diversity', 'odi_score',
    'evidence_robustness', 'priority_score'
]]

Embedding Asana chunks...
Clusters: 4


,cluster_id,cluster_size,avg_sentiment,satisfaction,source_type_diversity,odi_score,evidence_robustness,priority_score
0,1,124,-0.1083,0.4458,1.0000,0.1640,0.7536,0.3998
1,3,129,-0.1464,0.4268,0.8333,0.1765,0.6494,0.3657
2,0,120,-0.0734,0.4633,0.8333,0.1537,0.6419,0.3490
3,2,46,-0.0840,0.4580,0.3333,0.0595,0.2551,0.1377


### 2c. Revolut

Files available:
- `revolut_app/interview_revolut_01.txt` → `interview`
- `revolut/reviews_revolut.csv` (column: `content`) → `review`
- `revolut/tickets_revolut.csv` (column: `customer_message`) → `ticket`

In [8]:
revolut_chunks = []

# Interview (in revolut_app subfolder)
p = DATA / 'revolut_app' / 'interview_revolut_01.txt'
if p.exists():
    revolut_chunks += load_txt(str(p), 'interview')

# Reviews CSV
revolut_chunks += load_csv_text_col(
    str(DATA / 'revolut' / 'reviews_revolut.csv'), 'content', 'review', max_rows=80
)

# Tickets CSV — note: column is 'customer_message'
revolut_chunks += load_csv_ticket(
    str(DATA / 'revolut' / 'tickets_revolut.csv'), 'customer_message', max_rows=40
)

print(f'Revolut chunks: {len(revolut_chunks)}')
print(f'Source types present: {set(c["source_type"] for c in revolut_chunks)}')
print()
print('Note: Revolut dataset has fewer source types than Lidl/Asana.')
print('source_type_diversity will be lower — this is expected, not a bug.')

Revolut chunks: 86
Source types present: {'ticket', 'interview', 'review'}

Note: Revolut dataset has fewer source types than Lidl/Asana.
source_type_diversity will be lower — this is expected, not a bug.


In [9]:
print('Embedding Revolut chunks...')
revolut_emb = embed_chunks(revolut_chunks)
revolut_clusters = cluster(revolut_chunks, revolut_emb)
revolut_scored = score_clusters(revolut_clusters, revolut_chunks)

print(f'Clusters: {len(revolut_clusters)}')
pd.DataFrame(revolut_scored)[[
    'cluster_id', 'cluster_size', 'avg_sentiment',
    'satisfaction', 'source_type_diversity', 'odi_score',
    'evidence_robustness', 'priority_score'
]]

Embedding Revolut chunks...
Clusters: 3


,cluster_id,cluster_size,avg_sentiment,satisfaction,source_type_diversity,odi_score,evidence_robustness,priority_score
0,0,26,-0.2953,0.3523,0.5000,0.1958,0.4308,0.2898
1,2,33,0.0896,0.5448,0.3333,0.1747,0.3510,0.2452
2,1,27,0.1605,0.5802,0.3333,0.1318,0.3266,0.2097


## 3. Validation summary — pass/fail assessment

In [10]:
datasets = {
    'Lidl':    lidl_scored,
    'Asana':   asana_scored,
    'Revolut': revolut_scored,
}

STD_THRESHOLD = 0.10   # minimum odi_score std to pass
RANGE_THRESHOLD = 0.25 # minimum odi_score range to pass

print('=' * 65)
print(f'{"Dataset":<10} {"Clusters":>8} {"avg_sent std":>12} {"odi std":>8} {"odi range":>10} {"PASS?":>6}')
print('-' * 65)

results = {}
for name, scored in datasets.items():
    df = pd.DataFrame(scored)
    odi_std   = df['odi_score'].std()
    odi_range = df['odi_score'].max() - df['odi_score'].min()
    sent_std  = df['avg_sentiment'].std()
    passed    = (odi_std >= STD_THRESHOLD) and (odi_range >= RANGE_THRESHOLD)
    results[name] = {'odi_std': odi_std, 'odi_range': odi_range,
                     'sent_std': sent_std, 'passed': passed,
                     'n_clusters': len(df)}
    mark = '✅' if passed else '⚠️ '
    print(f'{name:<10} {len(df):>8} {sent_std:>12.4f} {odi_std:>8.4f} {odi_range:>10.4f} {mark:>6}')

print('=' * 65)
n_passed = sum(1 for r in results.values() if r['passed'])
overall  = '✅ PASS' if n_passed >= 2 else '❌ FAIL — check mapping'
print(f'\nOverall: {n_passed}/3 datasets pass → {overall}')
print(f'Thresholds: odi_score std ≥ {STD_THRESHOLD}, odi_score range ≥ {RANGE_THRESHOLD}')

Dataset    Clusters avg_sent std  odi std  odi range  PASS?
-----------------------------------------------------------------
Lidl              7       0.1423   0.0282     0.0700    ⚠️ 
Asana             4       0.0324   0.0534     0.1170    ⚠️ 
Revolut           3       0.2453   0.0326     0.0640    ⚠️ 

Overall: 0/3 datasets pass → ❌ FAIL — check mapping
Thresholds: odi_score std ≥ 0.1, odi_score range ≥ 0.25


In [11]:
# Detailed per-dataset stats for decisions.md
print('--- Full stats for decisions.md entry ---\n')
for name, scored in datasets.items():
    df = pd.DataFrame(scored)
    print(f'{name}:')
    for col in ['avg_sentiment', 'satisfaction', 'odi_score',
                'evidence_robustness', 'priority_score']:
        print(f'  {col:<25} min={df[col].min():.4f}  '
              f'max={df[col].max():.4f}  '
              f'std={df[col].std():.4f}  '
              f'mean={df[col].mean():.4f}')
    print()

--- Full stats for decisions.md entry ---

Lidl:
  avg_sentiment             min=-0.1479  max=0.2458  std=0.1423  mean=0.0638
  satisfaction              min=0.4260  max=0.6229  std=0.0712  mean=0.5319
  odi_score                 min=0.0270  max=0.0970  std=0.0282  mean=0.0677
  evidence_robustness       min=0.2370  max=0.5032  std=0.0967  mean=0.4060
  priority_score            min=0.1110  max=0.2562  std=0.0544  mean=0.2030

Asana:
  avg_sentiment             min=-0.1464  max=-0.0734  std=0.0324  mean=-0.1030
  satisfaction              min=0.4268  max=0.4633  std=0.0162  mean=0.4485
  odi_score                 min=0.0595  max=0.1765  std=0.0534  mean=0.1384
  evidence_robustness       min=0.2551  max=0.7536  std=0.2193  mean=0.5750
  priority_score            min=0.1377  max=0.3998  std=0.1188  mean=0.3130

Revolut:
  avg_sentiment             min=-0.2953  max=0.1605  std=0.2453  mean=-0.0151
  satisfaction              min=0.3523  max=0.5802  std=0.1226  mean=0.4924
  odi_score    

## 4. Diagnostic — inspect lxyuan raw output (run if any dataset fails)

If a dataset fails the spread check, run this cell to inspect the raw lxyuan
output on a sample of texts. Common issues:
- All scores clustering near neutral → model is over-predicting neutral class
- All scores clustering positive → `_score_to_compound` mapping error
- No spread → check that `framework='tf'` is set (PyTorch bus error on Apple Silicon
  can fall back to a broken path silently)

In [12]:
# Only run this if you need to diagnose a failure
from pipeline.odi_scorer import _sentiment, _score_to_compound

diagnostic_texts = [
    # Clearly negative
    'The app crashed and I lost all my work. This is completely unacceptable.',
    'Verification has been stuck for three days. I need access to my funds urgently.',
    'Support never responds. Absolute disaster of a product.',
    # Clearly positive
    'The new dashboard is clean and easy to navigate. Huge improvement.',
    'Customer support resolved my issue within minutes. Very impressed.',
    'The collaboration features work perfectly across our distributed team.',
    # Calm / neutral B2B (where VADER struggled most)
    'We evaluated three project management tools before choosing this one.',
    'The API documentation covers the main endpoints but lacks examples.',
    'Onboarding took two weeks for a team of twelve.',
]

raw_results = _sentiment(diagnostic_texts)

print(f'{"Compound":>9}  {"Label":>10}  {"Score":>7}  Text')
print('-' * 75)
for text, r in zip(diagnostic_texts, raw_results):
    compound = _score_to_compound(r)
    print(f'{compound:>+9.4f}  {r["label"]:>10}  {r["score"]:>7.4f}  {text[:55]}')

compounds = [_score_to_compound(r) for r in raw_results]
print(f'\nDiagnostic spread: {min(compounds):.4f} to {max(compounds):.4f}  '
      f'std={np.std(compounds):.4f}')
print('\nExpected: negatives in range -0.7 to -0.95, positives in range +0.7 to +0.95,')
print('neutral/calm B2B in range -0.3 to +0.3')

 Compound       Label    Score  Text
---------------------------------------------------------------------------
  -0.7943    negative   0.7943  The app crashed and I lost all my work. This is complet
  -0.5540    negative   0.5540  Verification has been stuck for three days. I need acce
  -0.4879    negative   0.4879  Support never responds. Absolute disaster of a product.
  +0.7607    positive   0.7607  The new dashboard is clean and easy to navigate. Huge i
  +0.8975    positive   0.8975  Customer support resolved my issue within minutes. Very
  +0.9493    positive   0.9493  The collaboration features work perfectly across our di
  +0.5002    positive   0.5002  We evaluated three project management tools before choo
  +0.0000     neutral   0.7100  The API documentation covers the main endpoints but lac
  +0.4572    positive   0.4572  Onboarding took two weeks for a team of twelve.

Diagnostic spread: -0.7943 to 0.9493  std=0.6314

Expected: negatives in range -0.7 to -0.95, positive

In [13]:
# Chunk-level sentiment spread — the true lxyuan signal check
from pipeline.odi_scorer import _batch_sentiment

for name, chunks in [('Lidl', lidl_chunks), ('Asana', asana_chunks), ('Revolut', revolut_chunks)]:
    texts = [c['text'] for c in chunks]
    compounds = _batch_sentiment(texts)
    arr = np.array(compounds)
    print(f'{name}: chunk-level avg_sentiment  '
          f'min={arr.min():.3f}  max={arr.max():.3f}  '
          f'std={arr.std():.3f}  mean={arr.mean():.3f}')

Lidl: chunk-level avg_sentiment  min=-0.910  max=0.982  std=0.562  mean=0.053
Asana: chunk-level avg_sentiment  min=-0.963  max=0.972  std=0.517  mean=-0.107
Revolut: chunk-level avg_sentiment  min=-0.927  max=0.982  std=0.642  mean=-0.005
